In [1]:
# Librerías
import pandas as pd
import numpy as np
from datetime import datetime

print("✓ Librerías cargadas correctamente")

✓ Librerías cargadas correctamente


In [5]:
# Carga de datos originales
clients_df = pd.read_csv('./Tarea/Clients.csv')
ventas_df = pd.read_csv('./Tarea/ventas.csv')
facturas_df = pd.read_csv('./Tarea/facturas.csv')


In [ ]:
ventas_df['monto'] = ventas_df['cantidad'] * ventas_df['precio_unitario'] * (1 - ventas_df['descuento_pct'] / 100)

In [39]:
ventas_df = ventas_df.merge(clients_df, on = "id_cliente", how = "left")

In [40]:
df = ventas_df.groupby(["id_cliente", "ciudad"])["monto"].sum().reset_index()
df.head(2)

,id_cliente,ciudad,monto
0,100000,Quito,176.4285
1,100002,Quito,1351.2545


In [42]:
# EJERCICIO 1
# Crea dos segmentos de clientes para identificar clientes que compran
# Más de $1000
# menos de $1000
(df["monto"]>1000).replace({True: "Más de $1000", False: "Menos de $1000"})
## Identifique cuantos clientes hay en cada segmento

0        Menos de $1000
1          Más de $1000
2          Más de $1000
3        Menos de $1000
4        Menos de $1000
              ...      
52995      Más de $1000
52996    Menos de $1000
52997    Menos de $1000
52998    Menos de $1000
52999    Menos de $1000
Name: monto, Length: 53000, dtype: object

In [ ]:
# Bins: 0,18,25, 35, 50, inf
0-18 
18-25
25-35
35-50
50-inf

In [51]:
#(100, 500]
#(500, inf]

df["SegmentoCut"] = pd.cut(df["monto"], 
                           bins = [0,100,500, np.inf], 
                           labels= ["<$100", "$100-$500", ">$500"],
                           right=False)
df

,id_cliente,ciudad,monto,SegmentoCut
0,100000,Quito,176.4285,$100-$500
1,100002,Quito,1351.2545,>$500
2,100011,Quito,3674.8815,>$500
3,100016,Cuenca,899.6840,>$500
4,100037,Manta,84.0300,<$100
...,...,...,...,...
52995,999936,Guayaquil,1821.3550,>$500
52996,999943,Quito,470.5290,$100-$500
52997,999968,Quito,213.1300,$100-$500
52998,999972,Guayaquil,692.9900,>$500


In [54]:
## Las reglas de segmentos: 
## cliente E: compra menos de $50
## cliente D: entre 50 y 100
## cliente C: entre 100 y 200
## cliente B: entre 200 y 500
# cliente A: más de 500

df["SegmentoABC"] = pd.cut(df["monto"], 
       bins = [0,50,100, 200,500, np.inf],
       labels = ["E", "D", "C", "B", "A"]
        )
df

,id_cliente,ciudad,monto,SegmentoCut,SegmentoABC
0,100000,Quito,176.4285,$100-$500,C
1,100002,Quito,1351.2545,>$500,A
2,100011,Quito,3674.8815,>$500,A
3,100016,Cuenca,899.6840,>$500,A
4,100037,Manta,84.0300,<$100,D
...,...,...,...,...,...
52995,999936,Guayaquil,1821.3550,>$500,A
52996,999943,Quito,470.5290,$100-$500,B
52997,999968,Quito,213.1300,$100-$500,B
52998,999972,Guayaquil,692.9900,>$500,A


In [ ]:
#[0,50,100, 200,500, np.inf]
def segmentos(x):
    if x>500:
        return "A"
    elif x>200: 
        return "B"
    elif x>100: 
        return "C"
    elif x>50:
        return "D"
    else:
        return "E"

In [64]:
df["monto"].apply(segmentos)

0        C
1        A
2        A
3        A
4        D
        ..
52995    A
52996    B
52997    B
52998    A
52999    A
Name: monto, Length: 53000, dtype: object

In [ ]:
## Las reglas de segmentos: 
## cliente E: compra menos de $50
## cliente D: entre 50 y 100
## cliente C: entre 100 y 200
## cliente B: entre 200 y 500
# cliente A: más de 500

# Pero solo para GYE y UIO
## Cliente B: entre 200 y 1000
## Cliente a: más de 1000

In [71]:
df["ciudad"].unique()

array(['Quito', 'Cuenca', 'Manta', 'Guayaquil'], dtype=object)

In [ ]:
def segmentos_dif(row):
    monto = row["monto"]
    ciudad = row["ciudad"]

    if monto>1000 and ciudad in ("Guayaquil", "Quito"):
        return "A"
    elif monto> 200 and ciudad in ("Guayaquil", "Quito"):
        return "B"
    elif monto>500:
        return "A"
    elif monto>200: 
        return "B"
    elif monto>100: 
        return "C"
    elif monto>50:
        return "D"
    else:
        return "E"

In [70]:
df.apply(segmentos_dif, axis = 1)

0        C
1        A
2        A
3        A
4        D
        ..
52995    A
52996    B
52997    B
52998    A
52999    A
Length: 53000, dtype: object

In [73]:
df["SegmentoABC"].value_counts()

SegmentoABC
A    23338
B    17745
C     8304
D     2641
E      972
Name: count, dtype: int64

In [84]:
df["SegmentosUniformes"] = pd.qcut(df["monto"], q=6, labels=["F","E","D","C","B","A"])
df.head(2)


,id_cliente,ciudad,monto,SegmentoCut,SegmentoABC,SegmentosUniformes
0,100000,Quito,176.4285,$100-$500,C,E
1,100002,Quito,1351.2545,>$500,A,A
